In [1]:
# ================================================================
# NOTEBOOK 15: CWQ DEVELOPMENT ONLINE EVALUATION
# Runtime: T4 GPU
#
# Policies:
#   1. Relation heuristic, fixed budget
#   2. Learned ranker, fixed budget
#   3. SetStop
#
# Existing trained models and thresholds are used.
# Results are saved after every completed question.
# Gold answers are used only for evaluation and diagnostics.
# ================================================================

import collections
import csv
import hashlib
import importlib
import importlib.metadata
import json
import os
import random
import re
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np


# ---------------------------- SETTINGS ----------------------------

ROOT = Path("/content/drive/MyDrive/SetStop_KGQA")
SEED = 42

CONTROLLER_BATCH = 64
ANSWER_BATCH = 128

CODE_VERSION = "cwq_dev_online_v1_0"

# Matches the existing WebQSP evaluation.
# The 3_plus weight is a ranking proxy, not an exact answer count.
GAIN_WEIGHTS = np.asarray(
    [0.0, 1.0, 2.0, 3.5],
    dtype=np.float64,
)

METHODS = [
    "relation_fixed_e2e",
    "model_fixed_e2e",
    "setstop_e2e",
]

METHOD_NAMES = {
    "relation_fixed_e2e": "Relation fixed",
    "model_fixed_e2e": "Learned ranker, fixed budget",
    "setstop_e2e": "SetStop",
}


# ------------------------- FILE UTILITIES -------------------------

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def save_json(path, obj):
    temporary = path.with_name(path.name + ".partial")

    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(
            obj,
            file,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )

    temporary.replace(path)


def file_hash(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(8 * 1024**2),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def fingerprint(obj):
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True).encode()
    ).hexdigest()


def verify_inventory(path, inventory):
    entry = next(
        (
            item
            for item in inventory["files"]
            if item["name"] == path.name
        ),
        None,
    )

    if (
        entry is None
        or path.stat().st_size != entry["bytes"]
        or file_hash(path) != entry["sha256"]
    ):
        raise ValueError(
            f"Inventory verification failed: {path}"
        )


def tokens(text):
    return set(
        re.findall(
            r"[a-z0-9]+",
            text.lower().replace("_", " "),
        )
    )


# -------------------------- GRAPH PARSING -------------------------

def parse_question(
    row,
    entity_to_id,
    relation_to_id,
    relation_names,
):
    def entity(value):
        if isinstance(value, dict):
            value = value.get(
                "kb_id",
                value.get("id", value.get("entity")),
            )

        if isinstance(value, int) and not isinstance(value, bool):
            result = (
                value
                if 0 <= value < len(entity_to_id)
                else None
            )
        else:
            result = (
                entity_to_id.get(value)
                if isinstance(value, str)
                else None
            )

        if result is None:
            raise ValueError(f"Unmapped entity: {value}")

        return result

    def relation(value):
        if isinstance(value, int) and not isinstance(value, bool):
            result = (
                value
                if 0 <= value < len(relation_names)
                else None
            )
        else:
            result = (
                relation_to_id.get(value)
                if isinstance(value, str)
                else None
            )

        if result is None:
            raise ValueError(f"Unmapped relation: {value}")

        return result

    qid = str(row["id"])
    question = row["question"].strip()
    subgraph = row["subgraph"]

    raw_edges = subgraph.get(
        "tuples",
        subgraph.get("triples"),
    )

    if not isinstance(raw_edges, list):
        raise ValueError(f"Missing graph triples: {qid}")

    nodes = {
        entity(value)
        for value in subgraph["entities"]
    }

    edges = set()

    for edge in raw_edges:
        if (
            not isinstance(edge, (list, tuple))
            or len(edge) != 3
        ):
            raise ValueError(f"Invalid graph edge: {qid}")

        edges.add((
            entity(edge[0]),
            relation(edge[1]),
            entity(edge[2]),
        ))

    nodes |= {
        node
        for source, rel, target in edges
        for node in (source, target)
    }

    topics = {
        entity(value)
        for value in row.get(
            "entities",
            row.get("topic_entities", []),
        )
    }

    gold = {
        entity(value)
        for value in row["answers"]
    }

    if not (
        topics
        and topics <= nodes
        and gold <= nodes
        and 2 <= len(gold) <= 50
    ):
        raise ValueError(
            f"CWQ question no longer satisfies eligibility: {qid}"
        )

    grouped = collections.defaultdict(
        lambda: collections.defaultdict(set)
    )

    for source, rel, target in sorted(edges):
        grouped[source][rel].add(target)

    adjacency = {
        source: {
            rel: sorted(targets)
            for rel, targets in group.items()
        }
        for source, group in grouped.items()
    }

    # Gold is kept separate from the graph passed to search.
    graph = {
        "qid": qid,
        "question": question,
        "topics": sorted(topics),
        "adjacency": adjacency,
        "input_edges": len(raw_edges),
        "unique_edges": len(edges),
        "graph_nodes": len(nodes),
    }

    return graph, gold


# ------------------------ FRONTIER RETRIEVAL ----------------------

def frontier(
    graph,
    depth,
    expanded,
    relation_tokens,
    max_hops,
):
    question_tokens = tokens(graph["question"])

    branches = [
        (source, rel, depth[source], targets)
        for source in sorted(depth)
        if depth[source] < max_hops
        for rel, targets in sorted(
            graph["adjacency"].get(source, {}).items()
        )
        if (source, rel) not in expanded
    ]

    # Same ordering used for CWQ preprocessing.
    return sorted(
        branches,
        key=lambda branch: (
            -len(
                question_tokens
                & relation_tokens[branch[1]]
            ),
            branch[2],
            -len(branch[3]),
            branch[1],
            branch[0],
        ),
    )


# --------------------------- MODEL INPUTS -------------------------

def controller_text(
    question,
    predicted_count,
    visited_count,
    step,
    branch,
    history,
    relations,
):
    history_text = (
        " -> ".join(history[-6:])
        if history
        else "none"
    )

    return (
        f"Question: {question}\n"
        f"Answers found so far: {predicted_count}\n"
        f"Visited entities: {visited_count}\n"
        f"Search step: {step}\n"
        f"Source depth: {branch[2]}\n"
        f"Candidate relation: {relations[branch[1]]}\n"
        f"Candidate target count: {len(branch[3])}\n"
        f"Recent relations: {history_text}"
    )


def answer_text(
    question,
    branch,
    source_path,
    target,
    adjacency,
    relations,
):
    path = source_path + [branch[1]]

    path_text = " -> ".join(
        relations[rel]
        for rel in path
    )

    return (
        f"Question: {question}\n"
        f"Candidate relation path: {path_text}\n"
        f"Candidate depth: {branch[2] + 1}\n"
        f"Incoming relation: {relations[branch[1]]}\n"
        f"Entities returned by final relation: {len(branch[3])}\n"
        f"Candidate outgoing relation count: "
        f"{len(adjacency.get(target, {}))}"
    )


# -------------------------- SEARCH ENGINE -------------------------
# This function has no gold-answer argument.

def run_search(
    graph,
    relations,
    relation_tokens,
    controller_predict,
    answer_predict,
    max_hops,
    frontier_limit,
    action_cap,
    controller_threshold,
    answer_threshold,
    learned,
):
    depth = {
        node: 0
        for node in graph["topics"]
    }

    paths = {
        node: []
        for node in graph["topics"]
    }

    expanded = set()
    predicted = set()

    node_scores = {}
    history = []
    trace = []

    actions = 0
    edges = 0
    truncations = 0
    controller_candidates = 0
    answer_candidates = 0

    adaptive = None

    def snapshot(reason):
        return {
            "actions": actions,
            "retrieved_edges": edges,
            "frontier_truncations": truncations,
            "controller_candidates_scored": controller_candidates,
            "answer_candidates_scored": answer_candidates,
            "predicted_answers": sorted(predicted),
            "visited_depths": [
                [node, node_depth]
                for node, node_depth in sorted(depth.items())
            ],
            "scored_entities": sorted(node_scores),
            "expanded_branches": [
                list(branch)
                for branch in sorted(expanded)
            ],
            "stop_reason": reason,
            "trace": [
                dict(item)
                for item in trace
            ],
        }

    while actions < action_cap:
        full_frontier = frontier(
            graph,
            depth,
            expanded,
            relation_tokens,
            max_hops,
        )

        if not full_frontier:
            break

        shortlist = full_frontier[:frontier_limit]

        truncations += int(
            len(full_frontier) > len(shortlist)
        )

        maximum_continue = None
        should_stop = False

        if learned:
            texts = [
                controller_text(
                    graph["question"],
                    len(predicted),
                    len(depth),
                    actions,
                    branch,
                    history,
                    relations,
                )
                for branch in shortlist
            ]

            probabilities = np.asarray(
                controller_predict(texts),
                dtype=np.float32,
            )

            if (
                probabilities.shape != (len(shortlist), 4)
                or not np.isfinite(probabilities).all()
            ):
                raise ValueError(
                    "Invalid controller output."
                )

            if (
                np.any(probabilities < 0)
                or not np.allclose(
                    probabilities.sum(axis=1),
                    1,
                    atol=1e-4,
                )
            ):
                raise ValueError(
                    "Controller probabilities do not sum to one."
                )

            controller_candidates += len(shortlist)

            maximum_continue = float(
                np.max(1.0 - probabilities[:, 0])
            )

            should_stop = (
                maximum_continue < controller_threshold
            )

            selected_index = int(
                np.argmax(
                    probabilities @ GAIN_WEIGHTS
                )
            )

        else:
            # The shortlist is already relation-heuristic ordered.
            selected_index = 0

        trace.append({
            "step": actions,
            "visited_count": len(depth),
            "predicted_answer_count": len(predicted),
            "full_frontier_count": len(full_frontier),
            "shortlist_count": len(shortlist),
            "max_continue_score": maximum_continue,
            "threshold_would_stop": bool(should_stop),
            "next_source": shortlist[selected_index][0],
            "next_relation": relations[
                shortlist[selected_index][1]
            ],
        })

        if learned and should_stop and adaptive is None:
            adaptive = snapshot("learned_stop")

        # Continue for the fixed-budget ranker.
        # SetStop is the saved prefix at its first stopping decision.
        branch = shortlist[selected_index]
        source, rel, source_depth, targets = branch

        source_path = list(paths[source])

        texts = [
            answer_text(
                graph["question"],
                branch,
                source_path,
                target,
                graph["adjacency"],
                relations,
            )
            for target in targets
        ]

        answer_probabilities = np.asarray(
            answer_predict(texts),
            dtype=np.float32,
        )

        if (
            answer_probabilities.shape != (len(targets),)
            or not np.isfinite(answer_probabilities).all()
        ):
            raise ValueError(
                "Invalid recognizer output."
            )

        if np.any(
            (answer_probabilities < 0)
            | (answer_probabilities > 1)
        ):
            raise ValueError(
                "Answer probabilities outside [0,1]."
            )

        expanded.add((source, rel))

        for target, probability in zip(
            targets,
            answer_probabilities,
        ):
            new_depth = source_depth + 1

            if (
                target not in depth
                or new_depth < depth[target]
            ):
                depth[target] = new_depth
                paths[target] = source_path + [rel]

            best_score = max(
                node_scores.get(target, 0.0),
                float(probability),
            )

            node_scores[target] = best_score

            if best_score >= answer_threshold:
                predicted.add(target)

        actions += 1
        edges += len(targets)
        answer_candidates += len(targets)
        history.append(relations[rel])

    fixed = snapshot(
        "action_budget"
        if actions >= action_cap
        else "empty_frontier"
    )

    if adaptive is None:
        adaptive = fixed

    return fixed, adaptive


# ---------------------- POST-SEARCH DIAGNOSTICS -------------------

def remaining_reachable(graph, state, max_hops):
    # Reachability through unexpanded branches.
    # This function does not use gold labels.
    expanded = {
        tuple(branch)
        for branch in state["expanded_branches"]
    }

    best_depth = {}
    queue = collections.deque()

    for source, source_depth in state["visited_depths"]:
        if source_depth >= max_hops:
            continue

        for rel, targets in graph["adjacency"].get(
            source,
            {},
        ).items():
            if (source, rel) in expanded:
                continue

            for target in targets:
                new_depth = source_depth + 1

                if (
                    target not in best_depth
                    or new_depth < best_depth[target]
                ):
                    best_depth[target] = new_depth
                    queue.append(target)

    while queue:
        node = queue.popleft()
        node_depth = best_depth[node]

        if node_depth >= max_hops:
            continue

        for rel, targets in graph["adjacency"].get(
            node,
            {},
        ).items():
            if (node, rel) in expanded:
                continue

            for target in targets:
                new_depth = node_depth + 1

                if (
                    target not in best_depth
                    or new_depth < best_depth[target]
                ):
                    best_depth[target] = new_depth
                    queue.append(target)

    return set(best_depth)


def evaluate_state(
    graph,
    gold,
    state,
    policy,
    max_hops,
):
    predicted = set(state["predicted_answers"])
    visited = {
        node
        for node, node_depth in state["visited_depths"]
    }
    scored = set(state["scored_entities"])

    if (
        not predicted <= visited
        or not predicted <= scored
    ):
        raise ValueError(
            "Predicted answer was not visited and scored."
        )

    tp = len(predicted & gold)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    visited_gold = gold & visited

    reachable_missing = (
        (gold - visited)
        & remaining_reachable(
            graph,
            state,
            max_hops,
        )
    )

    if fp == 0 and fn == 0:
        error_category = "exact"
    elif not predicted:
        error_category = "empty_prediction"
    elif fp == 0:
        error_category = "incomplete_only"
    elif fn == 0:
        error_category = "over_prediction_only"
    else:
        error_category = "mixed_fp_fn"

    return {
        "qid": graph["qid"],
        "policy": policy,
        "precision": tp / max(1, len(predicted)),
        "recall": tp / len(gold),
        "answer_f1": (
            2 * tp / (len(predicted) + len(gold))
        ),
        "exact_match": int(predicted == gold),
        "complete_recall": int(gold <= predicted),
        "true_positive_count": tp,
        "false_positive_count": fp,
        "false_negative_count": fn,
        "gold_answer_count": len(gold),
        "predicted_answer_count": len(predicted),
        "predicted_answers": sorted(predicted),
        "gold_answers": sorted(gold),
        "actions": state["actions"],
        "retrieved_edges": state["retrieved_edges"],
        "input_edges": graph["input_edges"],
        "fraction_of_input_graph_exposed": (
            state["retrieved_edges"]
            / max(1, graph["input_edges"])
        ),
        "visited_entities": len(visited),
        "visited_gold_count": len(visited_gold),
        "visited_gold_recall": (
            len(visited_gold) / len(gold)
        ),
        "missed_by_search_count": len(gold - visited),
        "visited_gold_not_predicted_count": len(
            visited_gold - predicted
        ),
        "recognizer_false_negative_count": len(
            (gold & scored) - predicted
        ),
        "visited_gold_unscored_count": len(
            visited_gold - scored
        ),
        "remaining_unvisited_reachable_gold_count": (
            len(reachable_missing)
        ),
        "premature_stop": bool(
            state["stop_reason"] == "learned_stop"
            and reachable_missing
        ),
        "incomplete_set": bool(fn),
        "empty_prediction": not predicted,
        "error_category": error_category,
        "stop_reason": state["stop_reason"],
        "frontier_truncations": state["frontier_truncations"],
        "controller_candidates_scored": (
            state["controller_candidates_scored"]
        ),
        "answer_candidates_scored": (
            state["answer_candidates_scored"]
        ),
    }


def evaluate_question(
    graph,
    gold,
    relations,
    relation_tokens,
    controller_predict,
    answer_predict,
    settings,
):
    search_settings = {
        "max_hops": settings["max_hops"],
        "frontier_limit": settings["frontier_limit"],
        "action_cap": settings["action_cap"],
        "controller_threshold": settings["controller_threshold"],
        "answer_threshold": settings["answer_threshold"],
    }

    fixed, adaptive = run_search(
        graph,
        relations,
        relation_tokens,
        controller_predict,
        answer_predict,
        **search_settings,
        learned=True,
    )

    relation, _ = run_search(
        graph,
        relations,
        relation_tokens,
        controller_predict,
        answer_predict,
        **search_settings,
        learned=False,
    )

    if (
        adaptive["actions"] > fixed["actions"]
        or adaptive["retrieved_edges"] > fixed["retrieved_edges"]
        or not set(adaptive["predicted_answers"])
        <= set(fixed["predicted_answers"])
    ):
        raise ValueError(
            "Shared-prefix consistency check failed."
        )

    states = {
        "relation_fixed_e2e": relation,
        "model_fixed_e2e": fixed,
        "setstop_e2e": adaptive,
    }

    records = [
        evaluate_state(
            graph,
            gold,
            states[policy],
            policy,
            settings["max_hops"],
        )
        for policy in METHODS
    ]

    if any(
        record["retrieved_edges"] > graph["unique_edges"]
        for record in records
    ):
        raise ValueError(
            "Retrieved-edge count exceeds the input graph."
        )

    return {
        "qid": graph["qid"],
        "question": graph["question"],
        "records": records,
        "traces": {
            policy: states[policy]["trace"]
            for policy in METHODS
        },
        "shared_prefix_check": True,
    }


# ------------------------- RESULT SUMMARY -------------------------

def summarize(records):
    summaries = {}

    for policy in METHODS:
        group = [
            record
            for record in records
            if record["policy"] == policy
        ]

        if not group:
            raise ValueError(f"No results for {policy}")

        tp, fp, fn = (
            sum(record[key] for record in group)
            for key in [
                "true_positive_count",
                "false_positive_count",
                "false_negative_count",
            ]
        )

        def mean(key):
            return float(np.mean([
                record[key]
                for record in group
            ]))

        result = {
            "method": METHOD_NAMES[policy],
            "questions": len(group),
            "macro_precision": mean("precision"),
            "macro_recall": mean("recall"),
            "macro_f1": mean("answer_f1"),
            "micro_f1": (
                2 * tp / max(1, 2 * tp + fp + fn)
            ),
            "exact_match": mean("exact_match"),
            "complete_recall": mean("complete_recall"),
            "mean_actions": mean("actions"),
            "mean_retrieved_edges": mean("retrieved_edges"),
            "mean_fraction_of_input_graph_exposed": mean(
                "fraction_of_input_graph_exposed"
            ),
            "visited_gold_recall": mean("visited_gold_recall"),
            "empty_prediction_rate": mean("empty_prediction"),
            "incomplete_set_rate": mean("incomplete_set"),
            "premature_stop_fraction_all_questions": mean(
                "premature_stop"
            ),
            "true_positive_count": tp,
            "false_positive_count": fp,
            "false_negative_count": fn,
            "stop_reasons": dict(
                collections.Counter(
                    record["stop_reason"]
                    for record in group
                )
            ),
        }

        learned_stops = [
            record
            for record in group
            if record["stop_reason"] == "learned_stop"
        ]

        result["premature_stop_rate_given_learned_stop"] = (
            sum(
                record["premature_stop"]
                for record in learned_stops
            ) / len(learned_stops)
            if learned_stops
            else None
        )

        summaries[policy] = result

    return summaries


# ------------------------------ MAIN ------------------------------

def main():
    from google.colab import drive
    drive.mount("/content/drive")

    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Select Runtime > Change runtime type > T4 GPU, "
            "then rerun."
        )

    os.environ["USE_TF"] = "0"
    os.environ["USE_FLAX"] = "0"

    # Reuse the exact HF package versions recorded during training.
    training_summary_path = (
        ROOT
        / "outputs/cwq_controller_training_v1/"
          "cwq_controller_training_summary_v1.json"
    )

    trained_versions = read_json(
        training_summary_path
    )["dependency_versions"]

    with tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".txt",
    ) as constraint:
        constraint.write(
            "torch==" + torch.__version__.split("+")[0] + "\n"
        )
        constraint.flush()

        requirements = [
            f"{package}=={trained_versions[package]}"
            for package in [
                "transformers",
                "peft",
                "accelerate",
                "bitsandbytes",
                "safetensors",
            ]
        ]

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "-c",
                constraint.name,
                *requirements,
            ],
            check=True,
        )

    importlib.invalidate_caches()

    import transformers
    import peft

    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        BitsAndBytesConfig,
    )
    from peft import (
        PeftModel,
        prepare_model_for_kbit_training,
    )
    from safetensors import safe_open

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    data = ROOT / "data/extracted/cwq"
    preprocessing = ROOT / "outputs/cwq_preprocessing_v1"
    controller_output = ROOT / "outputs/cwq_controller_training_v1"
    answer_output = ROOT / "outputs/cwq_answer_recognizer_v1"
    output = ROOT / "outputs/cwq_dev_online_v1"

    output.mkdir(parents=True, exist_ok=True)

    generation = read_json(
        preprocessing / "cwq_state_generation_summary_v1.json"
    )
    preprocessing_inventory = read_json(
        preprocessing / "artifact_inventory_v1.json"
    )

    verify_inventory(
        preprocessing / "cwq_state_generation_summary_v1.json",
        preprocessing_inventory,
    )

    manifest_path = Path(generation["manifest_path"])
    verify_inventory(
        manifest_path,
        preprocessing_inventory,
    )
    manifest = read_json(manifest_path)

    controller_inventory = read_json(
        controller_output / "artifact_inventory_v1.json"
    )
    answer_inventory = read_json(
        answer_output / "artifact_inventory_v1.json"
    )

    for name in [
        "cwq_controller_training_summary_v1.json",
        "cwq_dev_stop_threshold_v1.json",
    ]:
        verify_inventory(
            controller_output / name,
            controller_inventory,
        )

    for name in [
        "cwq_answer_recognizer_summary_v1.json",
        "cwq_answer_recognizer_threshold_v1.json",
    ]:
        verify_inventory(
            answer_output / name,
            answer_inventory,
        )

    controller_summary = read_json(
        controller_output / "cwq_controller_training_summary_v1.json"
    )
    stop_report = read_json(
        controller_output / "cwq_dev_stop_threshold_v1.json"
    )
    answer_summary = read_json(
        answer_output / "cwq_answer_recognizer_summary_v1.json"
    )
    answer_report = read_json(
        answer_output / "cwq_answer_recognizer_threshold_v1.json"
    )

    signatures = [
        generation["signature"],
        manifest["signature"],
        preprocessing_inventory["signature"],
        controller_summary["data_signature"],
        controller_inventory["data_signature"],
        answer_summary["data_signature"],
        answer_inventory["data_signature"],
    ]

    if (
        len(set(signatures)) != 1
        or stop_report["test_used"]
        or answer_report["test_used"]
    ):
        raise ValueError(
            "CWQ signatures or threshold provenance disagree."
        )

    adapter_dir = Path(controller_summary["adapter_path"])
    answer_dir = Path(answer_summary["saved_model"])

    for name in [
        "controller_config.json",
        "adapter_config.json",
        "adapter_model.safetensors",
    ]:
        verify_inventory(
            adapter_dir / name,
            controller_inventory,
        )

    for name in [
        "answer_recognizer_config.json",
        "config.json",
        "model.safetensors",
    ]:
        verify_inventory(
            answer_dir / name,
            answer_inventory,
        )

    controller_config = read_json(
        adapter_dir / "controller_config.json"
    )
    answer_config = read_json(
        answer_dir / "answer_recognizer_config.json"
    )

    if controller_config["class_to_id"] != {
        "0": 0,
        "1": 1,
        "2": 2,
        "3_plus": 3,
    }:
        raise ValueError(
            "Unexpected controller class order."
        )

    if answer_config["class_to_id"] != {
        "not_answer": 0,
        "answer": 1,
    }:
        raise ValueError(
            "Unexpected answer class order."
        )

    settings = {
        "max_hops": generation["configuration"]["max_hops"],
        "frontier_limit": generation["configuration"]["frontier_limit"],
        "action_cap": generation["configuration"]["action_cap"],
        "controller_threshold": float(
            stop_report["selected_threshold"]
        ),
        "answer_threshold": float(
            answer_report["selected_threshold"]
        ),
    }

    if not (
        settings["max_hops"]
        == controller_config["max_hops"]
        == answer_config["max_hops"]
        and settings["frontier_limit"]
        == controller_config["frontier_limit"]
        and settings["action_cap"]
        == controller_config["max_actions"]
        and settings["controller_threshold"]
        == answer_report["controller_threshold_unchanged"]
    ):
        raise ValueError(
            "Saved models do not match the CWQ protocol."
        )

    selected = manifest["splits"]["dev"]
    qids = [item["qid"] for item in selected]

    if (
        len(qids) != len(set(qids))
        or set(qids)
        != set(
            generation["configuration"]["selected_qids"]["dev"]
        )
    ):
        raise ValueError(
            "Development question selection mismatch."
        )

    audit = read_json(
        ROOT
        / "outputs/cwq_audit_v1/cwq_dataset_audit_summary_v1.json"
    )

    if (
        audit["signature"]
        != generation["configuration"]["audit_signature"]
    ):
        raise ValueError(
            "Source audit does not match Notebook 12."
        )

    source_files = audit["configuration"]["source_files"]

    print("=" * 90)
    print("NOTEBOOK 15: CWQ DEVELOPMENT ONLINE EVALUATION")
    print("=" * 90)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Transformers:", transformers.__version__)
    print("Development questions:", len(selected))
    print("Settings:", json.dumps(settings, indent=2))
    print("Inference only. Both thresholds remain unchanged.")

    for name in [
        "dev_simple.json",
        "entities.txt",
        "relations.txt",
    ]:
        print("Verifying source:", name, flush=True)

        expected = source_files[name]

        if (
            (data / name).stat().st_size != expected["bytes"]
            or file_hash(data / name) != expected["sha256"]
        ):
            raise ValueError(
                f"CWQ source changed: {name}"
            )

    model_identity = {
        "controller_adapter": file_hash(
            adapter_dir / "adapter_model.safetensors"
        ),
        "controller_configuration": file_hash(
            adapter_dir / "controller_config.json"
        ),
        "answer_model": file_hash(
            answer_dir / "model.safetensors"
        ),
        "answer_configuration": file_hash(
            answer_dir / "answer_recognizer_config.json"
        ),
    }

    protocol = {
        "code_version": CODE_VERSION,
        "data_signature": generation["signature"],
        "manifest_sha256": file_hash(manifest_path),
        "models": model_identity,
        "settings": settings,
        "development_qids": qids,
        "source_sha256": {
            name: source_files[name]["sha256"]
            for name in [
                "dev_simple.json",
                "entities.txt",
                "relations.txt",
            ]
        },
        "controller_batch": CONTROLLER_BATCH,
        "answer_batch": ANSWER_BATCH,
        "controller_max_length": (
            controller_config["maximum_sequence_length"]
        ),
        "answer_max_length": (
            answer_config["maximum_sequence_length"]
        ),
        "gain_class_weights": GAIN_WEIGHTS.tolist(),
        "answer_accumulation": (
            "Maximum probability across encountered paths; "
            "monotonic accepted set."
        ),
        "software_versions": {
            package: importlib.metadata.version(package)
            for package in [
                "transformers",
                "peft",
                "accelerate",
                "bitsandbytes",
                "safetensors",
            ]
        },
        "pytorch_version": torch.__version__,
        "test_used": False,
    }

    signature = fingerprint(protocol)

    cache = (
        output
        / "question_cache"
        / signature[:16]
    )
    cache.mkdir(parents=True, exist_ok=True)

    protocol_path = output / "cwq_dev_online_protocol_v1.json"

    save_json(
        protocol_path,
        {
            "signature": signature,
            **protocol,
        },
    )

    def cache_path(qid):
        name = hashlib.sha256(
            qid.encode()
        ).hexdigest()[:24]

        return cache / f"{name}.json"

    completed = {}

    for qid in qids:
        path = cache_path(qid)

        cached = (
            read_json(path)
            if path.exists()
            else None
        )

        if (
            cached
            and cached.get("signature") == signature
            and cached.get("qid") == qid
        ):
            payload = {
                key: value
                for key, value in cached.items()
                if key != "payload_sha256"
            }

            if (
                cached.get("payload_sha256")
                != fingerprint(payload)
            ):
                raise ValueError(
                    f"Cached result content hash failed: {qid}"
                )

            cached_policies = {
                record["policy"]
                for record in cached["records"]
            }

            if (
                cached_policies != set(METHODS)
                or len(cached["records"]) != len(METHODS)
            ):
                raise ValueError(
                    f"Invalid cached policies: {qid}"
                )

            completed[qid] = cached

    print(
        "Completed questions reused:",
        len(completed),
    )

    started = time.time()

    if len(completed) < len(selected):
        with open(
            data / "entities.txt",
            encoding="utf-8",
        ) as file:
            entity_to_id = {
                line.rstrip("\r\n"): index
                for index, line in enumerate(file)
            }

        with open(
            data / "relations.txt",
            encoding="utf-8",
        ) as file:
            relations = [
                line.rstrip("\r\n")
                for line in file
            ]

        relation_to_id = {
            name: index
            for index, name in enumerate(relations)
        }

        relation_tokens = [
            tokens(name)
            for name in relations
        ]

        quantization = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
            llm_int8_skip_modules=["score"],
        )

        tokenizer_controller = AutoTokenizer.from_pretrained(
            str(adapter_dir),
            local_files_only=True,
        )
        tokenizer_controller.padding_side = "right"
        tokenizer_controller.truncation_side = "right"

        base = AutoModelForSequenceClassification.from_pretrained(
            controller_config["base_model"],
            revision=controller_config["base_revision"],
            num_labels=4,
            id2label={
                0: "gain_0",
                1: "gain_1",
                2: "gain_2",
                3: "gain_3_plus",
            },
            label2id={
                "gain_0": 0,
                "gain_1": 1,
                "gain_2": 2,
                "gain_3_plus": 3,
            },
            problem_type="single_label_classification",
            quantization_config=quantization,
            dtype=torch.float16,
            device_map={"": 0},
            token=False,
            cache_dir="/content/setstop_cwq_hf_cache_v1",
            attn_implementation="sdpa",
        )

        base.config.pad_token_id = (
            tokenizer_controller.pad_token_id
        )
        base.config.use_cache = False

        base = prepare_model_for_kbit_training(
            base,
            use_gradient_checkpointing=False,
        )

        with safe_open(
            str(adapter_dir / "adapter_model.safetensors"),
            framework="pt",
            device="cpu",
        ) as adapter_file:
            if not any(
                "score" in key
                for key in adapter_file.keys()
            ):
                raise ValueError(
                    "Saved adapter lacks its trained classification head."
                )

        controller_model = PeftModel.from_pretrained(
            base,
            str(adapter_dir),
            is_trainable=False,
        )
        controller_model.eval()

        tokenizer_answer = AutoTokenizer.from_pretrained(
            str(answer_dir),
            local_files_only=True,
        )
        tokenizer_answer.padding_side = "right"
        tokenizer_answer.truncation_side = "right"

        answer_model = AutoModelForSequenceClassification.from_pretrained(
            str(answer_dir),
            local_files_only=True,
        )
        answer_model.to("cuda").eval()

        for model in [controller_model, answer_model]:
            for parameter in model.parameters():
                parameter.requires_grad_(False)

        def make_predictor(
            model,
            tokenizer,
            max_length,
            batch_size,
            classes,
        ):
            memo = {}

            def predict(texts):
                missing = list(dict.fromkeys(
                    text
                    for text in texts
                    if text not in memo
                ))

                order = sorted(
                    range(len(missing)),
                    key=lambda index: len(missing[index]),
                )

                with torch.inference_mode():
                    for start in range(
                        0,
                        len(order),
                        batch_size,
                    ):
                        indices = order[
                            start:start + batch_size
                        ]

                        batch_texts = [
                            missing[index]
                            for index in indices
                        ]

                        encoded = tokenizer(
                            batch_texts,
                            padding=True,
                            truncation=True,
                            max_length=max_length,
                            pad_to_multiple_of=8,
                            return_tensors="pt",
                        )

                        encoded = {
                            key: value.to("cuda")
                            for key, value in encoded.items()
                        }

                        with torch.autocast(
                            device_type="cuda",
                            dtype=torch.float16,
                        ):
                            logits = model(
                                **encoded,
                                return_dict=True,
                            ).logits

                        values = torch.softmax(
                            logits.float(),
                            dim=-1,
                        ).cpu().numpy()

                        if (
                            values.shape
                            != (len(batch_texts), classes)
                            or not np.isfinite(values).all()
                        ):
                            raise ValueError(
                                "Model inference produced invalid probabilities."
                            )

                        for text, value in zip(
                            batch_texts,
                            values,
                        ):
                            memo[text] = value.copy()

                values = np.asarray(
                    [memo[text] for text in texts],
                    dtype=np.float32,
                ).reshape(len(texts), classes)

                return (
                    values
                    if classes == 4
                    else values[:, 1]
                )

            return predict

        by_line = {
            int(entry["line_number"]): entry
            for entry in selected
        }

        if len(by_line) != len(selected):
            raise ValueError(
                "Duplicate selected development line number."
            )

        print(
            "Both trained models loaded. "
            "Starting development-only search...",
            flush=True,
        )

        with open(
            data / "dev_simple.json",
            encoding="utf-8",
        ) as file:
            for line_number, line in enumerate(file, 1):
                entry = by_line.get(line_number)

                if (
                    entry is None
                    or entry["qid"] in completed
                ):
                    continue

                raw = json.loads(line)

                question_hash = hashlib.sha256(
                    " ".join(
                        raw["question"].lower().split()
                    ).encode()
                ).hexdigest()

                if (
                    str(raw["id"]) != entry["qid"]
                    or question_hash
                    != entry["question_sha256"]
                ):
                    raise ValueError(
                        "Selected development question changed."
                    )

                graph, gold = parse_question(
                    raw,
                    entity_to_id,
                    relation_to_id,
                    relations,
                )

                if len(gold) != entry["gold_count"]:
                    raise ValueError(
                        "Selected gold count changed."
                    )

                # Memoization is reset for each question.
                controller_predict = make_predictor(
                    controller_model,
                    tokenizer_controller,
                    controller_config["maximum_sequence_length"],
                    CONTROLLER_BATCH,
                    4,
                )

                answer_predict = make_predictor(
                    answer_model,
                    tokenizer_answer,
                    answer_config["maximum_sequence_length"],
                    ANSWER_BATCH,
                    2,
                )

                result = evaluate_question(
                    graph,
                    gold,
                    relations,
                    relation_tokens,
                    controller_predict,
                    answer_predict,
                    settings,
                )

                result["signature"] = signature
                result["payload_sha256"] = fingerprint(result)

                save_json(
                    cache_path(graph["qid"]),
                    result,
                )
                completed[graph["qid"]] = result

                if (
                    len(completed) % 10 == 0
                    or len(completed) == len(selected)
                ):
                    print(
                        f"Questions: {len(completed):,}/"
                        f"{len(selected):,} | "
                        f"this invocation "
                        f"{(time.time() - started) / 60:.1f} min",
                        flush=True,
                    )

                del controller_predict, answer_predict

        if set(completed) != set(qids):
            raise ValueError(
                "Some development questions were not evaluated."
            )

    records = [
        record
        for qid in qids
        for record in completed[qid]["records"]
    ]

    summaries = summarize(records)

    per_question_path = (
        output / "cwq_dev_online_per_question_v1.jsonl"
    )
    temporary = per_question_path.with_name(
        per_question_path.name + ".partial"
    )

    with open(temporary, "w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record) + "\n")

    temporary.replace(per_question_path)

    per_policy_diagnostics = {}

    for policy in METHODS:
        group = [
            record
            for record in records
            if record["policy"] == policy
        ]

        diagnostic = {
            "error_categories": dict(
                collections.Counter(
                    record["error_category"]
                    for record in group
                )
            ),
            "gold_answers": sum(
                record["gold_answer_count"]
                for record in group
            ),
            "missed_by_search": sum(
                record["missed_by_search_count"]
                for record in group
            ),
            "visited_but_not_predicted": sum(
                record["visited_gold_not_predicted_count"]
                for record in group
            ),
            "scored_gold_rejected": sum(
                record["recognizer_false_negative_count"]
                for record in group
            ),
            "visited_gold_unscored": sum(
                record["visited_gold_unscored_count"]
                for record in group
            ),
            "premature_learned_stops": sum(
                record["premature_stop"]
                for record in group
            ),
            "questions_with_frontier_truncation": sum(
                record["frontier_truncations"] > 0
                for record in group
            ),
        }

        if (
            diagnostic["missed_by_search"]
            + diagnostic["visited_but_not_predicted"]
            != summaries[policy]["false_negative_count"]
        ):
            raise ValueError(
                "Search/recognition error decomposition failed."
            )

        per_policy_diagnostics[policy] = diagnostic

    setstop = summaries["setstop_e2e"]
    comparisons = {}

    for policy in [
        "relation_fixed_e2e",
        "model_fixed_e2e",
    ]:
        baseline = summaries[policy]

        comparisons[policy] = {
            "macro_f1_difference": (
                setstop["macro_f1"] - baseline["macro_f1"]
            ),
            "exact_match_difference": (
                setstop["exact_match"] - baseline["exact_match"]
            ),
            "complete_recall_difference": (
                setstop["complete_recall"]
                - baseline["complete_recall"]
            ),
            "action_reduction_percent": (
                100
                * (
                    baseline["mean_actions"]
                    - setstop["mean_actions"]
                )
                / max(1e-15, baseline["mean_actions"])
            ),
            "edge_reduction_percent": (
                100
                * (
                    baseline["mean_retrieved_edges"]
                    - setstop["mean_retrieved_edges"]
                )
                / max(
                    1e-15,
                    baseline["mean_retrieved_edges"],
                )
            ),
        }

    summary = {
        "evaluation": "CWQ development-only online evaluation",
        "signature": signature,
        "data_signature": generation["signature"],
        "development_questions": len(qids),
        "settings": settings,
        "methods": summaries,
        "setstop_minus_baselines": comparisons,
        "shared_prefix_checks_passed": all(
            completed[qid]["shared_prefix_check"]
            for qid in qids
        ),
        "gold_used_for_policy": False,
        "test_used": False,
        "transformers_version": transformers.__version__,
        "peft_version": peft.__version__,
        "pytorch_version": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "runtime_minutes_this_invocation": (
            time.time() - started
        ) / 60,
        "notes": [
            "Gold is used only after search for metrics and diagnostics.",
            (
                "Graph exposure is a structural proxy; branch "
                "metadata comes from the supplied subgraph."
            ),
            (
                "The 3_plus ranking weight is 3.5, matching "
                "the existing WebQSP evaluation."
            ),
            (
                "Shared-prefix inference is not a per-policy "
                "latency benchmark."
            ),
            (
                "Results concern a selected multi-answer development "
                "subset, not the official full-CWQ benchmark."
            ),
        ],
    }

    diagnostics = {
        "signature": signature,
        "development_questions": len(qids),
        "methods": per_policy_diagnostics,
        "definitions": {
            "missed_by_search": (
                "Gold entity was not visited."
            ),
            "scored_gold_rejected": (
                "Visited gold entity received recognizer "
                "scores but was not accepted."
            ),
            "visited_gold_unscored": (
                "Visited gold entity never appeared as "
                "a scored incoming-branch target."
            ),
            "premature_learned_stop": (
                "Threshold stopping occurred while an unvisited "
                "gold entity remained reachable through an "
                "unexpanded branch within the hop limit."
            ),
        },
        "test_used": False,
    }

    summary_path = output / "cwq_dev_online_summary_v1.json"
    diagnostics_path = (
        output / "cwq_dev_online_diagnostics_v1.json"
    )

    save_json(summary_path, summary)
    save_json(diagnostics_path, diagnostics)

    table_path = output / "cwq_dev_online_results_v1.csv"

    columns = [
        "method",
        "questions",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "micro_f1",
        "exact_match",
        "complete_recall",
        "mean_actions",
        "mean_retrieved_edges",
        "visited_gold_recall",
        "empty_prediction_rate",
    ]

    with open(
        table_path,
        "w",
        encoding="utf-8",
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=columns,
        )
        writer.writeheader()
        writer.writerows(
            {
                key: summaries[policy][key]
                for key in columns
            }
            for policy in METHODS
        )

    protocol_path = output / "cwq_dev_online_protocol_v1.json"
    inventory_path = output / "artifact_inventory_v1.json"

    save_json(
        inventory_path,
        {
            "signature": signature,
            "data_signature": generation["signature"],
            "files": [
                {
                    "name": path.name,
                    "path": str(path),
                    "bytes": path.stat().st_size,
                    "sha256": file_hash(path),
                }
                for path in [
                    summary_path,
                    diagnostics_path,
                    per_question_path,
                    table_path,
                    protocol_path,
                ]
            ],
        },
    )

    print("\n" + "=" * 100)
    print("CWQ DEVELOPMENT ONLINE RESULTS")
    print("=" * 100)
    print(
        f'{"Method":<29} '
        f'{"F1":>8} '
        f'{"Exact":>8} '
        f'{"Complete":>9} '
        f'{"Actions":>9} '
        f'{"Edges":>10}'
    )

    for policy in METHODS:
        result = summaries[policy]

        print(
            f'{result["method"]:<29} '
            f'{result["macro_f1"]:>8.4f} '
            f'{result["exact_match"]:>8.4f} '
            f'{result["complete_recall"]:>9.4f} '
            f'{result["mean_actions"]:>9.2f} '
            f'{result["mean_retrieved_edges"]:>10.2f}'
        )

    print(
        "\nSetStop minus baselines:",
        json.dumps(comparisons, indent=2),
    )

    print("\nSEND THESE FOUR REPORTS:")

    for path in [
        summary_path,
        per_question_path,
        diagnostics_path,
        inventory_path,
    ]:
        print(path)

    print(
        "\nDevelopment evaluation complete. "
        "Both thresholds remain unchanged."
    )


main()

Mounted at /content/drive
NOTEBOOK 15: CWQ DEVELOPMENT ONLINE EVALUATION
GPU: Tesla T4
Transformers: 5.17.0
Development questions: 520
Settings: {
  "max_hops": 4,
  "frontier_limit": 128,
  "action_cap": 24,
  "controller_threshold": 0.853,
  "answer_threshold": 0.96
}
Inference only. Both thresholds remain unchanged.
Verifying source: dev_simple.json
Verifying source: entities.txt
Verifying source: relations.txt
Completed questions reused: 0


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both trained models loaded. Starting development-only search...
Questions: 10/520 | this invocation 2.7 min
Questions: 20/520 | this invocation 5.9 min
Questions: 30/520 | this invocation 9.4 min
Questions: 40/520 | this invocation 12.0 min
Questions: 50/520 | this invocation 15.0 min
Questions: 60/520 | this invocation 17.3 min
Questions: 70/520 | this invocation 19.6 min
Questions: 80/520 | this invocation 22.4 min
Questions: 90/520 | this invocation 25.1 min
Questions: 100/520 | this invocation 27.8 min
Questions: 110/520 | this invocation 30.6 min
Questions: 120/520 | this invocation 34.2 min
Questions: 130/520 | this invocation 36.8 min
Questions: 140/520 | this invocation 39.7 min
Questions: 150/520 | this invocation 42.6 min
Questions: 160/520 | this invocation 45.5 min
Questions: 170/520 | this invocation 49.0 min
Questions: 180/520 | this invocation 51.9 min
Questions: 190/520 | this invocation 54.9 min
Questions: 200/520 | this invocation 57.8 min
Questions: 210/520 | this in